# FireSight pilot - evaluation only (no training)

Scores the P10/P25/P50 checkpoints on the `val.jsonl` calibration/report slices.
Fix for the training-run OOM: `predict(batch=8, half=True)` (the checkpoint inherits
`batch=64` from training args otherwise) and `torch.cuda.empty_cache()` between models.


In [ ]:
import subprocess, sys, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()
print("torch", torch.__version__, "gpus", torch.cuda.device_count())


In [ ]:
import importlib.metadata as md, json, os
os.environ["YOLO_AUTOINSTALL"] = "False"
os.environ["YOLO_VERBOSE"] = "False"
os.environ["YOLO_OFFLINE"] = "True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def ver(n):
    try: return md.version(n)
    except Exception: return None

if ver("ultralytics") != "8.4.147":
    !pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")}, flush=True)


In [ ]:
import os
from pathlib import Path

def find_root(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    assert hits, pattern
    return hits[0]

DATA = str(find_root("manifest_summary.json").parent)
CODE = str(find_root("pilot_train.py").parent)
W = find_root("run_summary.json")
WEIGHTS = str(W.parents[2])  # .../runs/<Pxx>/run_summary.json -> runs
print("DATA", DATA); print("CODE", CODE); print("WEIGHTS", WEIGHTS)
assert (Path(WEIGHTS) / "P10/weights/best.pt").exists()


In [ ]:
import gc, json, subprocess
import torch
from ultralytics import YOLO

def read_list(name):
    out = []
    for line in open(f"{DATA}/lists/{name}"):
        line = line.strip()
        if line:
            p = Path(line)
            out.append(p if p.is_absolute() else Path(DATA) / p)
    return out

paths = read_list("val_cal.txt") + read_list("val_report.txt")
print("val images:", len(paths), flush=True)

for frac in ("P10", "P25", "P50"):
    w = f"{WEIGHTS}/{frac}/weights/best.pt"
    print("=== predicting", frac, w, flush=True)
    model = YOLO(w)
    rows = []
    for res in model.predict(paths, imgsz=640, conf=0.01, batch=8, half=True,
                             verbose=False, stream=True):
        dets = []
        for b in res.boxes:
            dets.append({"label": "fire" if int(b.cls.item()) == 0 else "smoke",
                         "confidence": round(float(b.conf.item()), 6),
                         "box": [float(v) for v in b.xyxy[0].tolist()]})
        rows.append({"id": Path(res.path).stem, "detections": dets})
    preds = f"/kaggle/working/preds_{frac}.jsonl"
    with open(preds, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")
    del model
    gc.collect(); torch.cuda.empty_cache()
    subprocess.run(["python", f"{CODE}/pilot_eval.py", "--dataset", DATA, "--preds", preds,
                    "--model", frac, "--target-fpr", "0.02",
                    "--out", f"/kaggle/working/metrics_{frac}.json"], check=True)
    print("done", frac, flush=True)


In [ ]:
import glob, json
import matplotlib.pyplot as plt

rows = []
for frac in ("P10", "P25", "P50"):
    mp, rp = f"/kaggle/working/metrics_{frac}.json", f"{WEIGHTS}/{frac}/run_summary.json"
    if not (os.path.exists(mp) and os.path.exists(rp)):
        print("missing", frac); continue
    met, run = json.load(open(mp)), json.load(open(rp))
    c = met["splits"]["val"]["combined"]
    rows.append({"run": frac, "images": run["n_train"],
                 "images_per_s": run["throughput"].get("images_per_s"),
                 "s_per_epoch": run["throughput"].get("median_s_per_epoch"),
                 "recall": c["recall"], "ci": c["recall_95ci"],
                 "fpr_neg": met["splits"]["val"]["fpr_per_negative_image"]["fpr"],
                 "map50": met["splits"]["val"]["box_ap"]["fire"]["ap50"]})
print(f"{'run':5s} {'images':>7s} {'img/s':>7s} {'s/epoch':>8s} {'recall@FPR2%':>13s} {'95% CI':>20s} {'FPR/neg':>8s} {'mAP50':>7s}")
for r in rows:
    ci = f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]" if r["ci"] else "-"
    print(f"{r['run']:5s} {r['images']:7d} {r['images_per_s']:7.1f} {r['s_per_epoch']:8.1f} "
          f"{r['recall']:13.4f} {ci:>20s} {r['fpr_neg']:8.4f} {r['map50']:7.4f}")
json.dump(rows, open("/kaggle/working/pilot_summary.json", "w"), indent=2)

if rows:
    xs=[r["images"] for r in rows]; ys=[r["recall"] for r in rows]
    lo=[r["ci"][0] for r in rows]; hi=[r["ci"][1] for r in rows]
    plt.figure(figsize=(6,4))
    plt.errorbar(xs, ys, yerr=[[y-l for y,l in zip(ys,lo)],[h-y for y,h in zip(ys,hi)]], marker="o")
    plt.xlabel("training images"); plt.ylabel("val recall @ FPR<=2% / negative")
    plt.title("FireSight pilot learning curve"); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig("/kaggle/working/learning_curve.png", dpi=160); plt.show()


## Next
Download `pilot_summary.json`, `metrics_P*.json`, `learning_curve.png`; then run the
bench_R **DEV-only** confirmation (one run per checkpoint) before GO/TRIM/STOP.
